In [24]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.optim import Adam
from torchvision.datasets import FashionMNIST
from torch.utils.data import DataLoader
import numpy as np

In [25]:
#classe per definire la trasformazione lineare iniziale (embedding) delle patch

class PatchEmbedding(nn.Module):
    def __init__(self, n_channels, d_model, patch_size):
        super().__init__()
        self.n_channels = n_channels
        self.d_model = d_model
        self.patch_size = patch_size

        #Tramite una convoluzione con uno stride grande quanto la dimensione del filtro (quindi non ho overlap), estraggo le singole patch e faccio una trasformazione lineare tramite i filtri convoluzionali
        #Ogni singola patch diventa un vettore d_model x 1
        #RICORDARE CHE PER OGNI PATCH CI SONO D_MODEL FILTRI DA APPLICARE CHE DANNO OGNUNO UN SINGOLO VALORE
        self.linear_projection = nn.Conv2d(in_channels=self.n_channels, out_channels=self.d_model, kernel_size=self.patch_size, stride=self.patch_size)

    def forward(self, x):
        x = self.linear_projection(x) #(batch_size, channels, height, width) -> (batch_size, d_model, P_row, P_col)
        x = x.flatten(2) #(batch_size, d_model, P_row, P_col) -> (batch_size, d_model, num_patches)
        x = x.transpose(1, 2) #(batch_size, d_model, num_patches) -> (batch_size, num_patches, d_model)

        return x

In [26]:
#classe per aggiungere il Positional Encoding

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len):
        super().__init__()
        #si tratta del numero di patch + 1, ed è il numero di encodings da creare e poi sommare a tutti i token delle patch + al classification token
        self.max_seq_len = max_seq_len
        #aggiungo il classification token che è inizializzato randomicamente, ma è uguale per tutte le immagini
        #verrà appreso un unico token che in inferenza verrà aggiunto a tutte le immagini uguale, poi la MSA darà output diversi per ogni immagine
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        #creiamo il positional encoding
        pos_enc = torch.zeros(self.max_seq_len, d_model)

        #itero sui token
        for pos in range(max_seq_len):
            for i in range(d_model):
                #posizione pari
                if i%2 == 0:
                    pos_enc[pos][i] = np.sin(pos/(10000) ** (i/d_model))
                #posizione dispari
                else:
                    pos_enc[pos][i] = np.cos(pos/(10000) ** (i/d_model))

        #dato che il positional encoding non è addestrabile ma ne ho bisogno nel modello
        #lo salvo con register_buffer che lo mette nello state_dict
        #(metto unsqueeze perché così (max_seq_len, d_model) -> (1, max_seq_len, d_model) e lo posso sommare più facilmente alle patch)
        self.register_buffer("pos_enc", pos_enc.unsqueeze(0))

    def forward(self, x):
        #espando il singolo cls_token con la dimensione del batch, così da poterlo concatenare facilmente a tutte le immagini del batch
        tokens_exp = self.cls_token.expand(x.size()[0], -1, -1) #shape di tokens_exp: (batch_size, 1, d)
        #aggiungo i token cls (uguali) a tutte le immagini del batch, sulla prima dimensione perché diventa come un token di patch
        x = torch.cat((tokens_exp, x), dim = 1)
        #sommo i positional_encoding
        x = x + self.pos_enc

        return x

In [27]:
#classe per descrivere il funzionamento della singola attention head

class AttentionHead(nn.Module):
    def __init__(self, d_model, head_size):
        super().__init__()

        self.head_size = head_size
        #non prendo solo parte delle patch, semplicemente uso una dimensione più piccola in output
        self.query = nn.Linear(in_features=d_model, out_features=head_size)
        self.key = nn.Linear(in_features=d_model, out_features=head_size)
        self.value = nn.Linear(in_features=d_model, out_features=head_size)

    def forward(self, x):
        #calcolo delle query, key, value
        #X*W W:d_model*head_size, X:n_patches*d_model, output: n_patches*head_size
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        #Q * K^T : n_patches*head_size X head_size*n_patches = n_patches * n_patches
        attention = Q @ K.transpose(-2, -1)

        #scaling
        attention = attention / (self.head_size**0.5)

        #softmax lungo le colonne
        attention = torch.softmax(attention, dim = -1)

        #shape finale: n_patches*n_patches X n_patches*head_size = n_patches*head_size
        attention = attention @ V

        return attention


In [28]:
#classe per MSA

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.head_size = d_model // n_heads
        #proiezione tramite trasformazione lineare
        self.W_o = nn.Linear(in_features=d_model, out_features=d_model)
        #inizializzo le head
        self.heads = nn.ModuleList([AttentionHead(d_model, self.head_size) for _ in range(n_heads)])

    def forward(self, x):
        #calcolo attenzione per ogni head e concateno sulla dimensione della head_size, ottenendo di nuovo vettori lunghi d_model
        output = torch.cat([head(x) for head in self.heads], dim=-1)
        #proietto nel nuovo spazio sempre di dimensione d_model
        proj_out = self.W_o(output)

        return proj_out


In [29]:
#blocco Encoder

class TransformerEncoder(nn.Module):
    def __init__(self, d_model, n_heads, r_mlp=4):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads

        #layer Normalization 1
        self.ln1 = nn.LayerNorm(d_model)

        #MSA
        self.mhsa = MultiHeadSelfAttention(d_model, n_heads)

        #layer Normalization 2
        self.ln2 = nn.LayerNorm(d_model)

        #MLP
        self.mlp = nn.Sequential(
            nn.Linear(in_features=d_model, out_features=r_mlp*d_model),
            nn.GELU(),
            nn.Linear(in_features=r_mlp*d_model, out_features=d_model)
        )

    def forward(self, x):
        #residual connection
        out = x + self.mhsa(self.ln1(x))
        out = out + self.mlp(self.ln2(out))

        return out


In [30]:
#ViT

class VisionTransformer(nn.Module):
    def __init__(self, d_model, n_classes, img_size, patch_size, n_channels, n_heads, n_layers):
        super().__init__()

        assert img_size[0] % patch_size[0] == 0 and img_size[1] % patch_size[1] == 0, "dimensione delle patch sbagliata"
        assert d_model % n_heads == 0, "dimensione degli embedding sbagliata"

        self.d_model = d_model
        self.n_classes = n_classes
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_channels = n_channels
        self.n_heads = n_heads

        self.n_patches = (self.img_size[0] * self.img_size[1]) // (self.patch_size[0] * self.patch_size[1])
        self.max_seq_length = self.n_patches + 1

        self.patch_embedding = PatchEmbedding(self.n_channels, self.d_model, self.patch_size)
        self.positional_encoding = PositionalEncoding(self.d_model, self.max_seq_length)
        self.transformer_encoder = nn.Sequential(*[TransformerEncoder(d_model, n_heads, r_mlp=4) for _ in range(n_layers)])

        self.classifier = nn.Sequential(
            nn.Linear(in_features=d_model, out_features=n_classes),
            nn.LogSoftmax(dim=-1)
        )

    def forward(self, x):
        x = self.patch_embedding(x)
        x = self.positional_encoding(x)
        x = self.transformer_encoder(x)
        x = self.classifier(x[:, 0])
        return x

In [31]:
d_model = 384
n_classes = 10
img_size = (224, 224)
patch_size = (16, 16)
n_channels = 1
n_heads = 6
n_layers = 12
batch_size = 16
epochs = 2
alpha = 0.005

In [32]:
transform = T.Compose([T.Resize(img_size), T.ToTensor()])

#train_set = MNIST(root="./Data", train=True, download=True, transform=transform)
#test_set = MNIST(root="./Data", train=False, download=True, transform=transform)

train_set = FashionMNIST(root="C:\\Users\\giaco\\Desktop\\Code\\Data", train=True, transform=transform)
test_set = FashionMNIST(root="C:\\Users\\giaco\\Desktop\\Code\\Data", train=False, transform=transform)

train_loader = DataLoader(train_set, shuffle=True, batch_size=batch_size)
test_loader = DataLoader(test_set, shuffle=False, batch_size=batch_size)

In [33]:
from torchsummary import summary
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

ViT = VisionTransformer(d_model, n_classes, img_size, patch_size, n_channels, n_heads, n_layers).to(device)
optimizer = Adam(ViT.parameters(), lr=alpha)
criterion = nn.NLLLoss()

print(summary(ViT, (1, 224, 224)))


cuda
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1          [-1, 384, 14, 14]          98,688
    PatchEmbedding-2             [-1, 196, 384]               0
PositionalEncoding-3             [-1, 197, 384]               0
         LayerNorm-4             [-1, 197, 384]             768
            Linear-5              [-1, 197, 64]          24,640
            Linear-6              [-1, 197, 64]          24,640
            Linear-7              [-1, 197, 64]          24,640
     AttentionHead-8              [-1, 197, 64]               0
            Linear-9              [-1, 197, 64]          24,640
           Linear-10              [-1, 197, 64]          24,640
           Linear-11              [-1, 197, 64]          24,640
    AttentionHead-12              [-1, 197, 64]               0
           Linear-13              [-1, 197, 64]          24,640
           Linear-14              

In [34]:
from tqdm import tqdm

for epoch in range(epochs):
    train_loss = 0.0
    for i, (x, y) in enumerate(tqdm(train_loader)):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = ViT(x)
        loss = criterion(out, y)
        loss.backward()
        train_loss += loss.item()
        optimizer.step()
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss/len(train_loader):.4f}")

100%|██████████| 3750/3750 [04:57<00:00, 12.62it/s]


Epoch 1/2, Train Loss: 306.8159


 21%|██        | 778/3750 [00:57<03:39, 13.53it/s]


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import classification_report
y_preds = []
y_real = []
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        out = ViT(x)

        y_preds.extend(torch.argmax(out, dim=1).tolist())
        y_real.extend(y.tolist())

print(classification_report(y_preds, y_real))

              precision    recall  f1-score   support

           0       0.12      0.70      0.20       168
           1       0.40      0.76      0.53       530
           2       0.07      0.37      0.12       192
           3       0.33      0.32      0.33      1041
           4       0.15      0.23      0.18       637
           5       0.06      0.96      0.10        57
           6       0.83      0.17      0.28      4820
           7       0.65      0.65      0.65       991
           8       0.11      0.78      0.19       138
           9       0.77      0.54      0.63      1426

    accuracy                           0.35     10000
   macro avg       0.35      0.55      0.32     10000
weighted avg       0.64      0.35      0.37     10000

